# calorai — agentic urban heat-budget audit (live demo)

A webinar-style walkthrough: auth check → live heatmap → environmental parameters → the deterministic physics pipeline → a printable PDF report.

Every number below traces to a documented equation (`docs/physics-references.md`) and a real FortyGuard Temperature API layer.

In [ ]:
from dotenv import load_dotenv; load_dotenv()
import sys
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")

DISTRICT = "phoenix"      # any key from /api/districts
DATE     = "2026-07-15"   # catalog 2021-01-01 → now (+12 h ahead)
HOUR     = 14             # audit hour, local
THRESHOLD_C = 30.0
DATA_SOURCE = "auto"      # auto (live → mock fallback) | mock (offline)

In [ ]:
import os
from fortyguard import FortyGuardClient
key = os.getenv("FORTYGUARD_API_KEY")
print("API key present:", bool(key), "| mode:", "live" if key else "offline (mock)")
if key:
    usage = FortyGuardClient().fetch_api_key_usage()
    cs = usage.get("credit_summary", {}) or {}
    print("credits:", f'{cs.get("cycle_credits_used", 0):,}', "/",
          f'{cs.get("total_available_credits", 0):,}',
          f'({cs.get("cycle_usage_percentage", 0)}% used this cycle)')

## Step 1 — the physics pipeline

`AuditAgent` scopes the request, fetches heatmap + exceedance layers and the 24 h environmental series, solves the surface energy balance (Stefan–Boltzmann + Brutsaert sky + street-canyon trapping + force-restore storage + Priestley–Taylor latent), and returns a fully structured report.

In [ ]:
from calorai.agent import AuditAgent, AuditRequest
from calorai.data_source import resolve_source

_source, mode = resolve_source(DATA_SOURCE)
print("resolved data source:", mode)

report = AuditAgent(AuditRequest(DISTRICT, DATE, hour=HOUR, threshold_c=THRESHOLD_C,
                                 data_source=DATA_SOURCE, narrator_kind=None)).run(narrate=False)
print(report["one_liner"])

## Step 2 — key sections

Cause attribution, ranked interventions with quantified °C, retrofit economics, vulnerability score, and the facade advisor.

In [ ]:
a = report["attribution"]
print("Energy attribution (W/m²): solar", a["solar_flux"], "· longwave", a["longwave_flux"],
      "· convection", a["convection_flux"], "· storage", a["storage_flux"],
      "· latent", a["latent_flux"])
print("Shares: solar", a["solar_share"], "% · longwave", a["longwave_share"],
      "% · convection", a["convection_share"], "%")
print()
for i, iv in enumerate(report["interventions"], 1):
    print(f"{i}. {iv['name']}  → −{iv['delta_t_c']:.1f} °C peak")
roi = report["retrofit_roi"]
print()
print("Retrofit ROI (top intervention):", f'{roi["annual_savings_usd"]:,}',
      "USD/yr · payback", roi["payback_years"], "yr")

In [ ]:
v = report["vulnerability"]
print("Vulnerability score:", v["score"]["score"], "/ 100 —", v["score"]["band"])
print("Worker alert:", v["safety_alert"]["level"], "→", v["safety_alert"]["action"])
print()
print("Facade advisor (kWh/m²/day):",
      {r["orientation"]: r["load_kwh_m2_per_day"] for r in report["facade"]["ranking"]})

## Step 3 — printable deliverable

The same report rendered as a PDF with a table of contents and six figures (energy balance, cause attribution, diurnal curve, facade loads, intervention ΔT, vulnerability components).

In [ ]:
from calorai.report import build_pdf_report
path = build_pdf_report(report)
print("PDF written to", path)

## Done

The web app exposes the same pipeline: `GET /api/audit` (JSON), `GET /api/report` (PDF), plus the single-page UI at `/`.